# Build and benchmark an agent workflow

This offline walkthrough uses the deterministic `TwitterEnv` simulator to show:

1. a multi-tool autonomous-agent environment,
2. dense action rewards plus a terminal goal score,
3. delayed external feedback through `wait_for_engagement`,
4. replay-ready episode traces, and
5. policy benchmarking on a fixed task dataset.

The same design works for ticketing, browser, CRM, coding, or operations agents.

In [ ]:
from pathlib import Path

from examples.environment.twitter.env import make_env
from plural import Benchmark, JSONLSink, ScriptedPolicy, TaskDataset, TraceWriter
from plural.environments import verify_replay
from plural.tracing import ParsedAction

env = make_env()
print(f"{env.name}@{env.version}  fingerprint={env.fingerprint()[:12]}…")
print("actions:", [tool.function.name for tool in env.action_defs])
print("tasks:", [task.task_id for task in env.iter_tasks()])

## 1. Run a delayed-feedback task

The likes task requires two actions: publish a non-empty post, then advance the simulator with `wait_for_engagement`. The tweet receives a small dense action reward; the completed goal supplies the terminal score.

Because this is a tool-only environment, the base `apply_action()` dispatches tools while final `step()` owns tracing and lifecycle bookkeeping.

In [ ]:
likes_task = list(env.iter_tasks())[1]
env.reset(likes_task, model="engagement-policy")
first = env.step(ParsedAction(name="tweet", arguments={"text": "A useful project update"}))
second = env.step(ParsedAction(name="wait_for_engagement"))
rollout = env.close_episode()

print("step rewards:", first.reward, second.reward)
print("likes:", rollout.trace.final_state["likes_on_own_posts"])
print("terminal reward:", rollout.trace.outcome.reward)
print("stop:", rollout.trace.stop_reason)

## 2. Inspect and replay the episode trace

The trace contains pre-action observations, parsed actions, tool results, dense reward events, final state, terminal score, and provenance. Deterministic replay reapplies recorded actions to a fresh environment and checks that behavior still matches.

In [ ]:
for decision in rollout.trace.turns():
    action = decision.parsed_action[0]
    reward = sum(event.value for event in decision.reward_events)
    print(decision.turn, action.name, action.arguments, "dense_reward=", reward)

replay = verify_replay(make_env(), rollout.trace, task=likes_task)
print("replay matched:", replay.ok)
print("returns:", rollout.trace.returns(gamma=0.9, source="both"))

## 3. Benchmark an agent behavior

Here the benchmark compares an engagement policy with an inactive baseline on exactly the same versioned task. Factories create fresh policies and environments per case; `TraceWriter` persists the episode traces used for later analysis or training.

In [ ]:
dataset = TaskDataset(
    name="twitter-likes-notebook",
    version=env.version,
    tasks=[likes_task],
)
policies = {
    "engage": lambda: ScriptedPolicy(
        [
            ParsedAction(name="tweet", arguments={"text": "A useful project update"}),
            ParsedAction(name="wait_for_engagement"),
        ]
    ),
    "inactive": lambda: ScriptedPolicy(
        [ParsedAction(name="respond", arguments={"text": "Take no action"})]
    ),
}

out = Path(".plural/examples")
out.mkdir(parents=True, exist_ok=True)
writer = TraceWriter(JSONLSink(out / "twitter-notebook-episodes.jsonl"))
try:
    report = Benchmark.from_policies(
        make_env(),
        policies,
        concurrency=2,
        environment_factory=make_env,
        trace_writer=writer,
    ).run(dataset=dataset)
finally:
    writer.close()

print(report.to_markdown())

## Adapt this pattern to your task

1. Model the external system in typed `State` and expose a minimal policy briefing through `Observation`.
2. Implement each permitted operation as an `@tool`; return structured success/error payloads.
3. Represent external events explicitly—like `wait_for_engagement`—so traces remain deterministic and replayable.
4. Use `step_reward()` for immediate action feedback and scorers for delayed task success.
5. Put goals, seeds, and simulator configuration in stable `TaskData`; collect them in a versioned `TaskDataset`.
6. Benchmark fresh policy factories with `Benchmark.from_policies()`, persist episode traces, and inspect failures per case.
7. To evaluate hosted models, replace policy factories with `Benchmark(env, models=[...], client=Client())` while keeping the same environment and dataset.

See `env.py` for the full simulator and `run.py` for the reply-to-mentions agent.